<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/Sentence_transformer_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# 1. Sample dataset
documents = [
    """Artificial intelligence is a branch of computer science that aims to build
    systems capable of performing tasks that normally require human intelligence.
    These tasks include reasoning, learning, problem solving, and decision-making.
    AI is widely used in healthcare, finance, education, and automation.""",

    """Machine learning is a subset of artificial intelligence. It allows systems
    to learn patterns from data instead of being explicitly programmed for every task.
    Supervised learning, unsupervised learning, and reinforcement learning are common
    machine learning approaches used in modern applications.""",

    """Natural language processing enables computers to understand, interpret, and
    generate human language. It is used in chatbots, sentiment analysis, translation,
    summarization, and question answering systems. NLP combines linguistics with
    machine learning and deep learning techniques.""",

    """Vector databases are used to store embeddings and perform similarity search.
    Tools such as FAISS, Chroma, Milvus, and Weaviate are commonly used in retrieval
    systems. They help applications quickly find text chunks that are semantically
    similar to a user's query.""",

    """Chunking is the process of splitting a large document into smaller pieces.
    Common chunking strategies include fixed-size chunking, sliding window chunking,
    and recursive chunking. Good chunking improves retrieval quality in RAG systems
    because it keeps related information together while staying small enough for search."""
]

def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks
all_chunks = []
for doc in documents:
    chunks = chunk_text(doc, chunk_size=80, overlap=20)
    all_chunks.extend(chunks)

print("Total chunks created:", len(all_chunks))
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_chunks)

#faiss search
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# output top 3
def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])

    return results

query = "Why Faiss is used?"
results = retrieve(query, top_k=3)

print("\nQuery:", query)
print("\nRetrieved Passages:")
for i, passage in enumerate(results, 1):
    print(f"\n{i}. {passage}")

Total chunks created: 5


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: Why Faiss is used?

Retrieved Passages:

1. Vector databases are used to store embeddings and perform similarity search. Tools such as FAISS, Chroma, Milvus, and Weaviate are commonly used in retrieval systems. They help applications quickly find text chunks that are semantically similar to a user's query.

2. Natural language processing enables computers to understand, interpret, and generate human language. It is used in chatbots, sentiment analysis, translation, summarization, and question answering systems. NLP combines linguistics with machine learning and deep learning techniques.

3. Chunking is the process of splitting a large document into smaller pieces. Common chunking strategies include fixed-size chunking, sliding window chunking, and recursive chunking. Good chunking improves retrieval quality in RAG systems because it keeps related information together while staying small enough for search.
